# Machine Learning Classifiers

## Purpose
Framework for training, testing and tuning Perceptron and KNN classifiers
## Author:   Rula Khayrallah
Copyright ©  Rula Khayrallah
## Warning
**Do not make any modifications to this notebook.**
## Content
In this Jupyter notebook, we have the framework needed to implement classifiers to recognize images of handwritten digits and to identify Iris flower species based on some specific measurements.
## Installs and Imports

In [ ]:
!pip install import-ipynb
import import_ipynb
from example import Example
from urllib.parse import urljoin
import urllib.request
import numpy as np
DIGITS_URL = 'https://raw.githubusercontent.com/Rula-Khayrallah/28566Digits/refs/heads/master/'
IRIS_URL = 'https://raw.githubusercontent.com/Rula-Khayrallah/28566Iris/refs/heads/master/'

## Main Functions
### _classify_

In [ ]:
def classify(classifier, dataset, hyperparameter=1):   
    if not valid_arguments(classifier, dataset, hyperparameter):
        return
    classification_algorithm = globals()[classifier]
    classification_algorithm(dataset, hyperparameter)

def valid_arguments(classifier, dataset, hyperparameter):
    """
    Validate the arguments passed to the main function
    :param
    classifier: (string) classifier to use: perceptron or knn?
    dataset: (string) dataset to use: digits or iris?
    hyperparameter: (int)how many iterations for perceptron??  What value of k for knn?
    :return: Boolean
    """
    valid = True
    if hyperparameter <= 0:
        valid = False
        print("Invalid hyperparameter:", hyperparameter)
    if classifier not in {'knn', 'perceptron'} :
        valid = False
        print("Invalid classifier:", classifier)
    if dataset not in {'digits', 'iris'}:
        valid = False
        print("Dataset not supported:", dataset)
        
    return valid

### _perceptron_

In [ ]:
def perceptron(dataset, iterations):
    """
    Train, validate and test a Perceptron classifier on the given
    dataset.
    The validation output is saved in 'dataset_perceptron_Validation_output.txt'
    The test output is saved in 'dataset_perceptron_Test_output.txt'
    :param dataset: (string) digits or iris
    :param iterations: integer - number of iteration to use when
    training the perceptron.
    """
    print('Training...')
    data = get_data( dataset, "train")
    all_labels = tuple(sorted({example.label for example in data}))
    p = Perceptron(all_labels, iterations)
    p.train(data)

    for phase in ("Validation", "Test"):
        print(f'{phase} in progress...')
        data = get_data(dataset, phase.lower())
        predictions = [p.predict(f) for f in data]
        report(f'{dataset}_perceptron_{phase}_output.txt', predictions, data, phase)

### _knn_

In [ ]:
def knn(dataset, k):
    """
    Classify data based on the nearest k neighbors.
    The validation output is saved in 'dataset_knn_Validation_output.txt'
    The test output is saved in 'dataset_knn_Test_output.txt'
    :param dataset: (string) digits or iris
    :param k: number of neighbors
    """
    training_data = get_data( dataset, "train")
    for phase in ("Validation", "Test"):
        print(f'{phase} in progress...')
        data = get_data(dataset, phase.lower())
        predictions = [predict_knn(training_data, example, k)
                       for example in data]
        report(f'{dataset}_knn_{phase}_output.txt', predictions, data, phase)

## Input Functions
Functions to fetch, read and preprocess the data to create the Example objects 

In [ ]:
def get_data(dataset, phase):
    """
    Get the training/validation/test data from the files.
    :param dataset: (string) digits or iris
    :param phase: train, validation or test
    :return: a list of Example objects representing the data for the
    given dataset.
    """
    if dataset == "digits":
        images = urljoin(DIGITS_URL, f'{phase}images.txt')
        labels = urljoin(DIGITS_URL, f'{phase}labels.txt')
        data = read_digits(images, labels)
    elif dataset == "iris":
        source = urljoin(IRIS_URL, f'{phase}.csv')
        data = read_iris(source)
    return data

In [ ]:
def read_digits(images, labels):
    """
    Read the digits image and label files and build the
    example objects with the feature vectors.
    :param images: Name of the text file containing the images
    :param labels: Name of the text file containing the labels
    :return: A list of Example objects
    """
    IMAGE_SIZE = 28
    NUM_FEATURES = IMAGE_SIZE ** 2 + 1
    data = []
    try:
        with urllib.request.urlopen(labels) as label_file:
            label_data = label_file.read().decode('utf-8')
    except Exception as err:
        print(err, labels)
    else: 
        lines = label_data.splitlines()
        for  each_line in lines :                 
            example = Example(int(each_line.strip()))
            data.append(example)

    image_row = 0
    count = 0
    feature_vector = np.zeros(NUM_FEATURES, int)
    
    try:
        with urllib.request.urlopen(images) as image_file:
            image_data = image_file.read().decode('utf-8')
    except Exception as err:
        print(err, images)
    else:
        lines = image_data.splitlines()
        for  each_line in lines:
            if image_row == 0:
                feature_vector = np.zeros(NUM_FEATURES, int)
                feature_vector[0] = 1 # bias
                feature = 1
            for each_char in each_line[0:IMAGE_SIZE]:
                feature_vector[feature] = 0 if each_char == ' ' else 1
                feature += 1
            image_row = (image_row + 1) % IMAGE_SIZE
            if image_row == 0:
                data[count].fvector = feature_vector
                count += 1
        data[-1].fvector = feature_vector

    return data

In [ ]:
def read_iris(filename):
    """
    Read the Iris csv file and construct the Example objects.
    The file is assumed to contain 5 columns:
    sepal length, sepal width, petal length, petal width and training label.
    The first 4 columns will be used as features in the feature vector
    constructed.

    :param filename: Name of the csv file containing iris data
    :return: A list of Example objects
    """
    data = []
    labels = []
    try:
        with urllib.request.urlopen(filename) as data_file:
            decoded_data = data_file.read().decode('utf-8')
    except Exception as err:
        print(err, filename)
        
    else:
        lines = decoded_data.splitlines()
        for each_line in lines:
            example_data = each_line.strip().split(',')
            num_features = len(example_data)
            feature_vector = np.zeros(num_features, float)
            feature_vector[0] = 1 # bias
            for count in range(num_features - 1):
                feature = float(example_data[count])
                feature_vector[count + 1] = feature
            label = example_data[num_features - 1]
            example = Example(label, feature_vector)
            data.append(example)
    return data

## Reporting Functions

In [ ]:
def compute_accuracy(predictions, data):
    """
    Compute the accuracy as the percentage of correct predictions over
    the total number of examples processed
    :param predictions: A list of the predicted labels
    :param data: A list of the example objects
    """
    if data:
        total = len(data)
        correct = [predictions[i] == data[i].label
                   for i in range(total)].count(True)
        return correct / total * 100

In [ ]:
def report(filename, predictions, data, phase):
    """
    Write the predictions to the specified file and print the accuracy.
    :param filename (string): name of the output file
    :param predictions: list of predicted labels
    """
    with open(filename, 'w', encoding='utf-8') as outfile:
        for p in predictions:
            outfile.write(f'{p}\n')
    accuracy = compute_accuracy(predictions, data)
    print(f'{phase} accuracy: {accuracy:.2f}%')